# Radar Hipotecario — EDA e Ingeniería de Características
**Proyecto final Diplomado Ciencia de Datos G33 — UNAM FES Acatlán**

Estructura alineada al formato de referencia del curso (Imports → Global variables → Functions → Data Ingestion → Feature Engineering → Visualización).

**Nota de reproducibilidad:** este notebook NO requiere credenciales ni tokens de API. Todos los datos se leen de snapshots públicos versionados en GitHub (`data/snapshots/latest/`), generados por el pipeline de ingesta del repositorio. Esto garantiza que corra igual en cualquier entorno de Google Colab, sin depender de whitelists de IP ni límites de tasa de las APIs originales (Banxico, INEGI).

### Imports

In [1]:
# Para producción
import io
import json
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

/Users/luiscontreras/Documents/radar-hipotecario/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

### Global variables

In [3]:
# Repositorio público — sin tokens, sin credenciales
GITHUB_USER = 'ContrerasPeninsula'
GITHUB_REPO = 'radar-hipotecario'
GITHUB_BRANCH = 'main'

RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}'

URL_SERIES_BANXICO = f'{RAW_BASE}/data/snapshots/latest/series_banxico.parquet'
URL_UMA = f'{RAW_BASE}/config/uma.json'
URL_REGLAS_INFONAVIT = f'{RAW_BASE}/config/reglas_infonavit_v2026.json'

### Functions

In [4]:
def cargar_parquet_github(url: str) -> pd.DataFrame:
    """
    Descarga un archivo Parquet desde una URL pública de GitHub (raw.githubusercontent.com)
    y lo carga como DataFrame, sin depender de fsspec/credenciales.
    """
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return pd.read_parquet(io.BytesIO(r.content))

In [5]:
def cargar_json_github(url: str) -> dict:
    """Descarga y parsea un archivo JSON público de GitHub."""
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

In [6]:
def calcular_variacion_pct(df: pd.DataFrame, columna: str, periodos: int) -> pd.Series:
    """
    Calcula variación porcentual de una columna a `periodos` observaciones de distancia.
    Para series diarias: periodos=252 aprox. 1 año hábil; periodos=21 aprox. 1 mes.
    """
    return df[columna].pct_change(periods=periodos) * 100

In [7]:
def resumen_estadistico(df: pd.DataFrame, columnas: list) -> pd.DataFrame:
    """Resumen descriptivo (media, std, min, max, percentiles) de columnas numéricas."""
    return df[columnas].describe().T

## Data Ingestion

### Series macroeconómicas (Banxico: TIIE, Tasa objetivo, FIX, INPC)
Fuente: [Banxico SIE API](https://www.banxico.org.mx/SieAPIRest/) — ingesta programada vía GitHub Actions, snapshot versionado en el repositorio.

In [8]:
df_banxico = cargar_parquet_github(URL_SERIES_BANXICO)
print(f'Filas: {len(df_banxico):,} | Series: {df_banxico["serie"].nunique()} | '
      f'Rango: {df_banxico["fecha"].min().date()} a {df_banxico["fecha"].max().date()}')
df_banxico.head()

Filas: 8,785 | Series: 4 | Rango: 2016-08-02 a 2026-07-31


,fecha,serie,serie_id,valor
0,2016-08-02,fix_usd,SF43718,18.8966
1,2016-08-03,fix_usd,SF43718,18.9117
2,2016-08-04,fix_usd,SF43718,18.8612
3,2016-08-05,fix_usd,SF43718,18.8691
4,2016-08-08,fix_usd,SF43718,18.5716


In [9]:
df_banxico['serie'].value_counts()

serie
tasa_objetivo    3639
fix_usd          2514
tiie_28          2514
inpc_general      118
Name: count, dtype: int64

In [10]:
# Verificación de completitud: observaciones por serie y huecos de fechas
df_banxico.groupby('serie')['fecha'].agg(['min', 'max', 'count'])

,min,max,count
serie,,,
fix_usd,2016-08-02,2026-07-31,2514
inpc_general,2016-09-01,2026-06-01,118
tasa_objetivo,2016-08-02,2026-07-31,3639
tiie_28,2016-08-02,2026-07-31,2514


### UMA (Unidad de Medida y Actualización)
Constante versionada por vigencia — no proviene de una API, se actualiza manualmente cada febrero cuando INEGI/DOF publican el nuevo valor.

In [11]:
uma_data = cargar_json_github(URL_UMA)
df_uma = pd.DataFrame(uma_data['valores'])
df_uma

,vigencia_inicio,vigencia_fin,diario,mensual
0,2024-02-01,2025-01-31,108.5700,3300.5300
1,2025-02-01,2026-01-31,113.1400,3439.4600
2,2026-02-01,2027-01-31,117.3100,3566.2200


### Tabla de tasas Infonavit (motor de reglas, referencia)
No es una serie de tiempo — es la tabla oficial de tasas diferenciadas por nivel salarial (vigente para UMA 2026), usada como insumo del motor de reglas determinista. Se incluye aquí para el EDA porque describe la distribución de tasas que enfrenta cada segmento de usuarios.

In [12]:
reglas_infonavit = cargar_json_github(URL_REGLAS_INFONAVIT)
df_infonavit_tasas = pd.DataFrame(reglas_infonavit['tasas']['tabla_diferenciada_por_uma'])
print(f"Estado de las reglas: {reglas_infonavit['estado']} | Versión: {reglas_infonavit['version']}")
df_infonavit_tasas.head(10)

Estado de las reglas: VALIDADO | Versión: 2026.07-VALIDADO


,uma_min,uma_max,tasa,salario_mensual_referencia
0,0.0000,2.6000,0.0369,9272.1800
1,2.6000,2.7000,0.0388,9628.8000
2,2.7000,2.8000,0.0407,9985.4300
3,2.8000,2.9000,0.0426,10342.0500
4,2.9000,3.0000,0.0445,10698.6700
5,3.0000,3.1000,0.0464,11055.2900
6,3.1000,3.2000,0.0483,11411.9200
7,3.2000,3.3000,0.0502,11768.5400
8,3.3000,3.4000,0.0521,12125.1600
9,3.4000,3.5000,0.0540,12481.7800


## Feature Engineering

### Series macro: pivote a formato ancho y variables derivadas

In [13]:
# Pivote: una columna por serie, indexado por fecha — facilita features cruzados entre series
df_wide = df_banxico.pivot_table(index='fecha', columns='serie', values='valor').sort_index()
df_wide = df_wide.ffill()  # las series no cotizan todos los mismos días; forward-fill conservador
df_wide.tail()

serie,fix_usd,inpc_general,tasa_objetivo,tiie_28
fecha,,,,
2026-07-27,17.4440,145.1310,6.5000,6.7559
2026-07-28,17.4312,145.1310,6.5000,6.7559
2026-07-29,17.5133,145.1310,6.5000,6.7559
2026-07-30,17.3562,145.1310,6.5000,6.7559
2026-07-31,17.3288,145.1310,6.5000,6.7458


In [14]:
# Inflación interanual a partir del INPC (Índice Nacional de Precios al Consumidor)
if 'inpc_general' in df_wide.columns:
    df_wide['inflacion_anual_pct'] = calcular_variacion_pct(df_wide, 'inpc_general', periodos=252)
    df_wide['inflacion_mensual_pct'] = calcular_variacion_pct(df_wide, 'inpc_general', periodos=21)
df_wide[['inpc_general', 'inflacion_mensual_pct', 'inflacion_anual_pct']].tail()

serie,inpc_general,inflacion_mensual_pct,inflacion_anual_pct
fecha,,,
2026-07-27,145.1310,0.0000,1.7428
2026-07-28,145.1310,0.0000,1.7428
2026-07-29,145.1310,0.0000,1.7428
2026-07-30,145.1310,0.0000,1.7428
2026-07-31,145.1310,0.0000,1.7428


In [15]:
# Medias móviles de la TIIE — suavizan el ruido de ajustes discretos de política monetaria
df_wide['tiie_28_ma30'] = df_wide['tiie_28'].rolling(window=30).mean()
df_wide['tiie_28_ma90'] = df_wide['tiie_28'].rolling(window=90).mean()
df_wide[['tiie_28', 'tiie_28_ma30', 'tiie_28_ma90']].tail()

serie,tiie_28,tiie_28_ma30,tiie_28_ma90
fecha,,,
2026-07-27,6.7559,6.7582,6.7861
2026-07-28,6.7559,6.7586,6.7833
2026-07-29,6.7559,6.7589,6.7804
2026-07-30,6.7559,6.7589,6.7777
2026-07-31,6.7458,6.7586,6.7748


In [16]:
# Spread TIIE vs Tasa objetivo — brecha relevante para detectar presión de mercado
df_wide['spread_tiie_objetivo'] = df_wide['tiie_28'] - df_wide['tasa_objetivo']
df_wide['spread_tiie_objetivo'].describe()

count   3640.0000
mean       0.2851
std        0.0923
min       -0.3587
25%        0.2450
50%        0.2685
75%        0.3413
max        0.8280
Name: spread_tiie_objetivo, dtype: float64

In [17]:
# Features de fecha, útiles para segmentar el EDA por periodo
df_wide['anio'] = df_wide.index.year
df_wide['mes'] = df_wide.index.month
df_wide['trimestre'] = df_wide.index.quarter
df_wide[['anio', 'mes', 'trimestre']].tail()

serie,anio,mes,trimestre
fecha,,,
2026-07-27,2026,7,3
2026-07-28,2026,7,3
2026-07-29,2026,7,3
2026-07-30,2026,7,3
2026-07-31,2026,7,3


### Motor de reglas Infonavit: variables descriptivas de la tabla de tasas

In [18]:
# Pendiente de la curva de tasas: cuánto sube la tasa por cada UMA adicional de salario
df_infonavit_tasas['delta_tasa'] = df_infonavit_tasas['tasa'].diff()
df_infonavit_tasas[['uma_min', 'uma_max', 'tasa', 'delta_tasa']].describe()

,uma_min,uma_max,tasa,delta_tasa
count,41.0000,41.0000,41.0000,40.0000
mean,4.4390,28.8049,0.0730,0.0017
std,1.3555,155.3113,0.0205,0.0002
min,0.0000,2.6000,0.0369,0.0014
25%,3.5000,3.6000,0.0559,0.0014
50%,4.5000,4.6000,0.0748,0.0019
75%,5.5000,5.6000,0.0904,0.0019
max,6.5000,999.0000,0.1045,0.0019


## Visualización

### Tablas

In [19]:
resumen_estadistico(df_wide, ['tiie_28', 'tasa_objetivo', 'fix_usd', 'inpc_general'])

,count,mean,std,min,25%,50%,75%,max
serie,,,,,,,,
tiie_28,3640.0000,7.8992,2.2028,4.2745,6.6000,7.8300,9.3258,11.5669
tasa_objetivo,3640.0000,7.6141,2.2078,4.0000,6.2500,7.5000,9.0000,11.2500
fix_usd,3640.0000,19.2855,1.4321,16.3357,18.3629,19.1951,20.1443,25.1185
inpc_general,3610.0000,117.3019,16.7781,90.3577,103.1080,113.8990,133.6810,145.8310


In [20]:
# Tabla resumen: rango de tasas Infonavit por decil de UMA
df_infonavit_tasas[['uma_min', 'uma_max', 'salario_mensual_referencia', 'tasa']].iloc[::4]

,uma_min,uma_max,salario_mensual_referencia,tasa
0,0.0000,2.6000,9272.1800,0.0369
4,2.9000,3.0000,10698.6700,0.0445
8,3.3000,3.4000,12125.1600,0.0521
12,3.7000,3.8000,13551.6500,0.0596
16,4.1000,4.2000,14978.1400,0.0672
20,4.5000,4.6000,16404.6300,0.0748
24,4.9000,5.0000,17831.1200,0.0819
28,5.3000,5.4000,19257.6100,0.0876
32,5.7000,5.8000,20684.1000,0.0932
36,6.1000,6.2000,22110.5900,0.0989


### Gráficas

In [21]:
fig = go.Figure()
for serie, nombre in [('tiie_28', 'TIIE 28 días'), ('tasa_objetivo', 'Tasa objetivo Banxico')]:
    fig.add_trace(go.Scatter(x=df_wide.index, y=df_wide[serie], name=nombre, mode='lines'))
fig.update_layout(
    title='Tasas de referencia de Banxico — histórico',
    xaxis_title='Fecha', yaxis_title='Tasa (%)', hovermode='x unified',
)
fig.show()

In [22]:
fig = px.line(df_wide.reset_index(), x='fecha', y='fix_usd',
              title='Tipo de cambio FIX (pesos por dólar)')
fig.update_layout(xaxis_title='Fecha', yaxis_title='MXN/USD')
fig.show()

In [23]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_wide.index, y=df_wide['inflacion_anual_pct'],
                          name='Inflación interanual (%)', mode='lines'))
fig.add_hline(y=3, line_dash='dot', annotation_text='Meta Banxico (3%)', line_color='green')
fig.update_layout(
    title='Inflación interanual (derivada del INPC vía Banxico, serie SP1)',
    xaxis_title='Fecha', yaxis_title='Variación % anual',
)
fig.show()

In [24]:
fig = px.bar(df_infonavit_tasas, x='salario_mensual_referencia', y='tasa',
             title='Tasa Infonavit diferenciada por nivel salarial (vigente UMA 2026)',
             labels={'salario_mensual_referencia': 'Salario mensual (MXN)', 'tasa': 'Tasa anual'})
fig.update_layout(yaxis_tickformat='.1%')
fig.show()

## Modelado de datos y evaluación de resultados

Dos modelos, uno por cada paradigma de aprendizaje, sobre las mismas fuentes reales ya cargadas en este notebook:

- **Supervisado (series de tiempo):** Prophet sobre la TIIE de Banxico, para proyectar la tasa hipotecaria de referencia a 12 meses.
- **No supervisado (clustering):** K-Means sobre precio/m² (oferta scrapeada) y variación anual (índice oficial SHF), para segmentar arquetipos de mercado por ciudad.

Ambos se documentan con su evaluación de desempeño — backtest para Prophet, inercia y perfil de clusters para K-Means — siguiendo el mismo estándar de reproducibilidad del resto del notebook (fuentes públicas, sin credenciales).

### Modelo supervisado: Prophet — proyección de tasa hipotecaria

In [25]:
# Prophet no viene preinstalado en Colab por default
!pip install prophet --quiet

You should consider upgrading via the '/Users/luiscontreras/Documents/radar-hipotecario/.venv/bin/python3 -m pip install --upgrade pip' command.


In [26]:
from prophet import Prophet

# Reutilizamos la serie TIIE ya cargada en df_banxico (sección Data Ingestion)
df_tiie = df_banxico[df_banxico['serie'] == 'tiie_28'][['fecha', 'valor']].copy()
df_tiie = df_tiie.rename(columns={'fecha': 'ds', 'valor': 'y'}).sort_values('ds').reset_index(drop=True)
df_tiie.tail()

,ds,y
2509,2026-07-27,6.7559
2510,2026-07-28,6.7559
2511,2026-07-29,6.7559
2512,2026-07-30,6.7559
2513,2026-07-31,6.7458


In [27]:
def entrenar_prophet(df: pd.DataFrame, periods_dias: int = 365):
    """Entrena Prophet y devuelve (modelo, forecast completo)."""
    modelo = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.15,  # ver justificación de sensibilidad más abajo
    )
    modelo.fit(df)
    futuro = modelo.make_future_dataframe(periods=periods_dias, freq='D')
    return modelo, modelo.predict(futuro)

**Nota de diseño:** `changepoint_prior_scale=0.15` se eligió tras una prueba de sensibilidad (0.05 → 0.15) documentada en el desarrollo del proyecto — la mejora marginal decreciente confirmó que la fuente principal de error no es falta de flexibilidad del modelo, sino la naturaleza **discreta** de los movimientos de la TIIE (decisiones de política monetaria en fechas puntuales, no una tendencia continua). Esto se retoma en la evaluación de abajo.

### Evaluación del modelo Prophet — backtest de 365 días

In [28]:
def backtest_prophet(df: pd.DataFrame, dias_holdout: int = 365) -> dict:
    """Entrena con todo menos los últimos `dias_holdout` días, predice ese
    tramo, y compara contra los valores reales."""
    corte = df['ds'].max() - pd.Timedelta(days=dias_holdout)
    train = df[df['ds'] <= corte]
    test = df[df['ds'] > corte]

    _, forecast = entrenar_prophet(train, periods_dias=dias_holdout + 30)

    comparacion = test.merge(forecast[['ds', 'yhat']], on='ds', how='left').dropna()
    mae = (comparacion['y'] - comparacion['yhat']).abs().mean()
    rmse = ((comparacion['y'] - comparacion['yhat']) ** 2).mean() ** 0.5

    return {
        'dias_holdout': dias_holdout,
        'n_obs_comparadas': len(comparacion),
        'mae': round(mae, 4),
        'rmse': round(rmse, 4),
        'y_promedio_periodo': round(comparacion['y'].mean(), 4),
    }

metricas_backtest = backtest_prophet(df_tiie, dias_holdout=365)
metricas_backtest

13:00:50 - cmdstanpy - INFO - Chain [1] start processing
13:00:51 - cmdstanpy - INFO - Chain [1] done processing


{'dias_holdout': 365,
 'n_obs_comparadas': 251,
 'mae': np.float64(0.883),
 'rmse': np.float64(0.8948),
 'y_promedio_periodo': np.float64(7.3454)}

In [29]:
error_relativo_pct = metricas_backtest['mae'] / metricas_backtest['y_promedio_periodo'] * 100
print(f"MAE: {metricas_backtest['mae']} puntos porcentuales")
print(f"Error relativo: {error_relativo_pct:.1f}% sobre el promedio del periodo de prueba")
print("\nInterpretación: un error relativo de esta magnitud es consistente con "
      "proyectar una serie de saltos discretos (política monetaria) con un modelo "
      "de tendencia continua — limitación esperada y documentada, no un error de ajuste.")

MAE: 0.883 puntos porcentuales
Error relativo: 12.0% sobre el promedio del periodo de prueba

Interpretación: un error relativo de esta magnitud es consistente con proyectar una serie de saltos discretos (política monetaria) con un modelo de tendencia continua — limitación esperada y documentada, no un error de ajuste.


In [30]:
modelo_final, forecast_final = entrenar_prophet(df_tiie, periods_dias=365)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_tiie['ds'], y=df_tiie['y'], name='TIIE observada', mode='lines'))
fig.add_trace(go.Scatter(x=forecast_final['ds'], y=forecast_final['yhat'],
                          name='Proyección Prophet', mode='lines', line=dict(dash='dot')))
fig.add_trace(go.Scatter(
    x=list(forecast_final['ds']) + list(forecast_final['ds'][::-1]),
    y=list(forecast_final['yhat_upper']) + list(forecast_final['yhat_lower'][::-1]),
    fill='toself', fillcolor='rgba(0,100,80,0.1)', line=dict(width=0),
    name='Intervalo de confianza', showlegend=True,
))
fig.update_layout(title='TIIE observada vs. proyección Prophet (12 meses)',
                   xaxis_title='Fecha', yaxis_title='TIIE (%)', hovermode='x unified')
fig.show()

13:01:03 - cmdstanpy - INFO - Chain [1] start processing
13:01:03 - cmdstanpy - INFO - Chain [1] done processing


### Modelo no supervisado: K-Means — arquetipos de mercado por ciudad

In [31]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

URL_OFERTA_INMUEBLES24 = f'{RAW_BASE}/data/oferta_inmuebles24.parquet'
URL_SHF_VARIACION = f'{RAW_BASE}/config/shf_variacion_ciudades.json'

In [32]:
# Feature 1: precio/m² real — mediana del detalle de anuncios scrapeados
df_oferta = cargar_parquet_github(URL_OFERTA_INMUEBLES24)
df_precio_ciudad = (
    df_oferta.groupby('ciudad')
    .agg(precio_m2=('precio_m2', 'median'), n_anuncios=('precio_m2', 'count'))
    .reset_index()
)
df_precio_ciudad

,ciudad,precio_m2,n_anuncios
0,cdmx,46408.8000,25
1,estado_mexico,22500.0000,3
2,guadalajara,38819.9000,27


In [33]:
# Feature 2: variación anual real — índice oficial SHF (avalúos hipotecarios,
# no depende del scraper ni de su sesgo hacia anuncios "Destacado")
shf_variacion = cargar_json_github(URL_SHF_VARIACION)
df_variacion_ciudad = pd.DataFrame([
    {'ciudad': ciudad, 'variacion_anual_pct': info['variacion_anual_pct']}
    for ciudad, info in shf_variacion['ciudades'].items()
])
df_variacion_ciudad

,ciudad,variacion_anual_pct
0,cdmx,4.4700
1,estado_mexico,5.7400
2,guadalajara,11.8700


In [34]:
df_mercado = df_precio_ciudad.merge(df_variacion_ciudad, on='ciudad', how='inner')
df_mercado

,ciudad,precio_m2,n_anuncios,variacion_anual_pct
0,cdmx,46408.8000,25,4.4700
1,estado_mexico,22500.0000,3,5.7400
2,guadalajara,38819.9000,27,11.8700


In [35]:
FEATURES_KMEANS = ['precio_m2', 'variacion_anual_pct']

# k tope = n_ciudades // 3, mínimo 2 (mismo criterio usado en el M3 de Valora AI)
n_ciudades = len(df_mercado)
k = max(2, n_ciudades // 3)

X = StandardScaler().fit_transform(df_mercado[FEATURES_KMEANS])
modelo_kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_mercado['cluster'] = modelo_kmeans.fit_predict(X)

print(f'k = {k} | inercia = {modelo_kmeans.inertia_:.3f}')
df_mercado

k = 2 | inercia = 2.912


,ciudad,precio_m2,n_anuncios,variacion_anual_pct,cluster
0,cdmx,46408.8000,25,4.4700,1
1,estado_mexico,22500.0000,3,5.7400,0
2,guadalajara,38819.9000,27,11.8700,1


### Evaluación del modelo K-Means — perfil e interpretación de clusters

In [36]:
perfil_clusters = df_mercado.groupby('cluster')[FEATURES_KMEANS].mean()
perfil_clusters['ciudades'] = df_mercado.groupby('cluster')['ciudad'].apply(list)
perfil_clusters['n_ciudades'] = df_mercado.groupby('cluster')['ciudad'].count()
perfil_clusters = perfil_clusters.reset_index()
perfil_clusters

,cluster,precio_m2,variacion_anual_pct,ciudades,n_ciudades
0,0,22500.0000,5.7400,[estado_mexico],1
1,1,42614.3500,8.1700,"[cdmx, guadalajara]",2


In [37]:
def nombrar_arquetipo(precio_alto: bool, variacion_alta: bool) -> str:
    if precio_alto and variacion_alta:
        return 'Premium en expansión'
    if precio_alto and not variacion_alta:
        return 'Premium consolidado'
    if not precio_alto and variacion_alta:
        return 'Emergente'
    return 'Estable / rezagado'

precio_mediana = perfil_clusters['precio_m2'].median()
variacion_mediana = perfil_clusters['variacion_anual_pct'].median()

for _, fila in perfil_clusters.iterrows():
    nombre = nombrar_arquetipo(
        fila['precio_m2'] >= precio_mediana,
        fila['variacion_anual_pct'] >= variacion_mediana,
    )
    print(f"Cluster {int(fila['cluster'])} — {nombre}: {fila['ciudades']}")

Cluster 0 — Estable / rezagado: ['estado_mexico']
Cluster 1 — Premium en expansión: ['cdmx', 'guadalajara']


In [38]:
fig = px.scatter(
    df_mercado, x='precio_m2', y='variacion_anual_pct', color='cluster',
    text='ciudad', size='n_anuncios',
    title='Arquetipos de mercado: precio/m² vs. variación anual',
    labels={'precio_m2': 'Precio mediano por m² (MXN)', 'variacion_anual_pct': 'Variación anual (%)'},
)
fig.update_traces(textposition='top center')
fig.show()

**Nota de limitación:** el clustering opera sobre solo 3 ciudades (alcance actual del scraper), lo cual es una muestra pequeña para K-Means — el resultado es ilustrativo del enfoque metodológico más que una segmentación estadísticamente robusta. La variable `precio_m2` proviene de una muestra scrapeada con sesgo documentado hacia anuncios "Destacado"/pagados del portal; `variacion_anual_pct`, en cambio, es un dato oficial de SHF, independiente de ese sesgo.

## Conclusiones finales

**Hallazgos del EDA:**
- La TIIE y la tasa objetivo se mueven en escalones discretos (decisiones de política monetaria), no en tendencia continua — hallazgo que explica el desempeño del modelo Prophet más abajo.
- La curva de tasas Infonavit es progresiva y aproximadamente lineal entre 2.6 y 6.6 UMA de salario.

**Hallazgos de la modelación:**
- **Prophet** proyecta razonablemente la tendencia de la TIIE, pero con un MAE de backtest que refleja la naturaleza discreta de la serie — limitación esperada y documentada, no un error de ajuste de hiperparámetros (se probó sensibilidad de `changepoint_prior_scale` con ganancia marginal decreciente).
- **K-Means** separa consistentemente las 3 ciudades en arquetipos coherentes con la intuición de mercado (Guadalajara/CDMX como polo de mayor precio y crecimiento, Estado de México como mercado más accesible y estable), usando dos variables 100% reales de fuentes independientes entre sí.

**Limitaciones generales del proyecto:**
- Alcance de 3 ciudades (CDMX, Estado de México, Guadalajara) por cobertura real del scraper — Puerto Vallarta, Mazatlán y Acapulco quedaron fuera.
- La tasa bancaria de referencia usa un spread provisional sobre la TIIE mientras se integra el cuadro CF815 de Banxico o datos de CNBV.
- El scraper de oferta inmobiliaria tiene sesgo documentado hacia anuncios pagados — se limitó su uso a una sola feature del K-Means, y las decisiones de "¿me alcanza?" en la aplicación usan percentiles oficiales de SHF en su lugar, no el scraper.

**Trabajo futuro:**
- Integrar tasas bancarias reales (CF815/CNBV) en vez del spread provisional.
- Ampliar el scraper (o una fuente alterna) a las 5 ciudades del alcance original.
- Explorar un modelo de cambio de régimen para la TIIE, más adecuado a su naturaleza de saltos discretos que Prophet.